# ValuSense — Phase 5 : De l'Agent IA à la Plateforme de Valorisation
## Corrections du Modèle + Roadmap Agent + Moteur de Calcul

### Analyse des Résultats de la Phase 4

| Observation | Diagnostic | Action |
|-------------|-----------|--------|
| **Accuracy 99%** | Les features structurelles sont les mêmes variables que les règles expertes — le modèle les a apprises parfaitement | Acceptable pour un PFE, mais ajouter des features continues (vol, spreads) pour que le modèle apporte une valeur au-delà des règles |
| **IFRS post-processing ↓ 20%** | La règle R1 est trop agressive — elle force Mark-to-Market sur tout actif Level 1 + liquide, y compris des actions qui devraient rester en DDM/Relative | Corriger R1 : Mark-to-Market seulement si le modèle ML n'a PAS déjà prédit une méthode plus spécifique |
| **4 modèles quasi-identiques** | Avec des features quasi-déterministes, RF ≈ XGBoost ≈ CatBoost | Normal — le différenciateur sera la couche agent, pas le modèle ML |

### Architecture Cible : Agent IA Complet

```
┌──────────────────────────────────────────────────────────────────────┐
│                      VALUSENSE AI AGENT                              │
│                                                                      │
│  ┌────────────────────────────────────────────────────────────────┐  │
│  │                    COUCHE LLM (Claude API)                     │  │
│  │  • Comprend la description en langage naturel                 │  │
│  │  • Extrait les features financières                           │  │
│  │  • Orchestre les outils via function calling                   │  │
│  └────────────┬──────────────────┬────────────────┬──────────────┘  │
│               │                  │                │                  │
│       ┌───────▼──────┐   ┌──────▼───────┐  ┌─────▼──────────┐      │
│       │  TOOL 1:     │   │  TOOL 2:     │  │  TOOL 3:       │      │
│       │  Recommend   │   │  Calculate   │  │  Explain       │      │
│       │  Method      │   │  Fair Value  │  │  Decision      │      │
│       │  (XGBoost +  │   │  (BSM, DCF,  │  │  (SHAP +       │      │
│       │   IFRS 13)   │   │   MC, DDM,   │  │   IFRS ref)    │      │
│       │              │   │   Binomial)  │  │                │      │
│       └──────────────┘   └─────────────┘  └────────────────┘      │
│                                                                      │
│  ┌────────────────────────────────────────────────────────────────┐  │
│  │                    COUCHE DONNÉES                               │  │
│  │  • yfinance (prix temps réel)                                 │  │
│  │  • FRED API (taux, courbes, spreads)                          │  │
│  │  • FinanceDatabase (métadonnées instruments)                  │  │
│  └────────────────────────────────────────────────────────────────┘  │
└──────────────────────────────────────────────────────────────────────┘
```


---
## 1. Correction de la Couche IFRS 13

La règle R1 originale force Mark-to-Market pour **tout** actif Level 1 + liquide.
C'est incorrect : une action Coca-Cola est Level 1 et liquide, mais sa juste valeur
pour un analyste est calculée par DDM ou DCF, pas simplement le cours de bourse.

**Mark-to-Market signifie** : "la juste valeur EST le prix de marché, sans modèle."
Ce n'est approprié que quand aucune méthode plus spécifique n'est pertinente.


In [5]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

MODELS_DIR = Path("models")

def apply_ifrs_constraints_v2(y_pred, X_df, le_target):
    """
    IFRS 13 post-processing V2 — regles corrigees.
    
    Changement principal : R1 ne force Mark-to-Market QUE si le modele ML
    a predit une methode inappropriee pour un actif Level 1.
    Si le ML a predit DDM, Relative, ou DCF pour un actif Level 1, c'est valide.
    """
    y_corrected = y_pred.copy()
    overrides = []
    
    for i in range(len(y_corrected)):
        row = X_df.iloc[i]
        method = le_target.inverse_transform([y_corrected[i]])[0]
        original = method
        
        # ── R1 CORRIGEE : Level 1 → Mark-to-Market SEULEMENT si la methode
        #    ML est inappropriee (ex: Monte-Carlo, Credit-Model sur un actif
        #    tres liquide sans optionalite ni risque de credit)
        if (row.get("ifrs_level", 0) == 1 and
            row.get("has_market_price", 0) == 1 and
            row.get("liquidity", 0) >= 2):
            # Methodes valides meme pour Level 1 — ne PAS overrider
            valid_for_l1 = {"DDM", "Relative", "DCF", "Mark-to-Market",
                "Cost-of-Carry", "Forward-Pricing",
                "Black-Scholes", "Binomial-Tree", "Monte-Carlo"}
            # Methodes qui n'ont pas de sens sur un actif Level 1 liquide
            if method not in valid_for_l1:
                method = "Mark-to-Market"
        
        # ── R2 : Level 3 + no market price → jamais Mark-to-Market
        if (row.get("ifrs_level", 0) == 3 and
            row.get("has_market_price", 0) == 0 and
            method == "Mark-to-Market"):
            if row.get("has_cash_flows", 0) == 1:
                method = "DCF"
            elif row.get("has_options_features", 0) == 1:
                method = "Monte-Carlo"
            else:
                method = "DCF"
        
        # ── R3 : Early exercise + BSM → Binomial-Tree
        if row.get("has_early_exercise", 0) == 1 and method == "Black-Scholes":
            method = "Binomial-Tree"
        
        # ── R4 : Path-dependent → Monte-Carlo
        if row.get("is_path_dependent", 0) == 1 and method != "Monte-Carlo":
            method = "Monte-Carlo"
        
        # ── R5 : Non-option asset + options method → DCF
        if (row.get("has_options_features", 0) == 0 and
            row.get("has_cash_flows", 0) == 1 and
            method in ["Black-Scholes", "Binomial-Tree"]):
            method = "DCF"
        
        # ── R6 : No cash flows + no options + DDM → fallback
        if (row.get("has_cash_flows", 0) == 0 and
            row.get("has_options_features", 0) == 0 and
            method == "DDM"):
            method = "Cost-of-Carry" if row.get("convenience_yield", 0) > 0 else "Forward-Pricing"
        
        if method != original:
            y_corrected[i] = le_target.transform([method])[0]
            overrides.append({"idx": i, "from": original, "to": method})
    
    return y_corrected, len(overrides), overrides

print("apply_ifrs_constraints_v2() — regle R1 corrigee")
print("  R1 ne force plus MtM sur DDM/Relative/DCF/Cost-of-Carry/Forward-Pricing")

apply_ifrs_constraints_v2() — regle R1 corrigee
  R1 ne force plus MtM sur DDM/Relative/DCF/Cost-of-Carry/Forward-Pricing


In [6]:
# ── Tester la V2 vs V1 ─────────────────────────────────────────
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score

le_target = joblib.load(MODELS_DIR / "label_encoder_target.pkl")
best_model = joblib.load(MODELS_DIR / "xgboost_valuation_recommender.pkl")
X_val = joblib.load(MODELS_DIR / "X_val.pkl")
y_val = joblib.load(MODELS_DIR / "y_val.pkl")

y_pred = best_model.predict(X_val)

# V2
y_ifrs_v2, n_v2, details_v2 = apply_ifrs_constraints_v2(y_pred, X_val, le_target)

print(f"IFRS V2 : {n_v2} predictions modifiees ({n_v2/len(y_pred)*100:.1f}%)")

corrections_v2 = {}
for o in details_v2:
    key = f"{o['from']} -> {o['to']}"
    corrections_v2[key] = corrections_v2.get(key, 0) + 1
if corrections_v2:
    for k, v in sorted(corrections_v2.items(), key=lambda x: -x[1]):
        print(f"  {k} : {v}")

# Comparaison
print(f"\n{'Metrique':20s} {'Sans IFRS':>12s} {'IFRS V2':>12s} {'Delta':>10s}")
print("-" * 56)
for name, fn in [("accuracy", accuracy_score), ("f1_weighted", lambda y,p: f1_score(y,p,average="weighted")),
                  ("f1_macro", lambda y,p: f1_score(y,p,average="macro")), ("kappa", cohen_kappa_score)]:
    before = fn(y_val, y_pred)
    after  = fn(y_val, y_ifrs_v2)
    delta  = after - before
    arrow  = "+" if delta > 0 else "" if delta == 0 else ""
    print(f"  {name:18s} {before:12.4f} {after:12.4f} {delta:+10.4f}")

# Sauvegarder la V2
joblib.dump(apply_ifrs_constraints_v2, MODELS_DIR / "ifrs_constraints_v2.pkl")

IFRS V2 : 1 predictions modifiees (0.0%)
  Credit-Model -> Mark-to-Market : 1

Metrique                Sans IFRS      IFRS V2      Delta
--------------------------------------------------------
  accuracy                 0.9905       0.9902    -0.0003
  f1_weighted              0.9906       0.9902    -0.0003
  f1_macro                 0.9877       0.9871    -0.0006
  kappa                    0.9889       0.9886    -0.0004


['models\\ifrs_constraints_v2.pkl']

---
## 2. Moteurs de Calcul de Valorisation

C'est l'élément clé qui transforme ValuSense d'un simple classificateur
en un **outil de valorisation complet**. Chaque méthode recommandée par le ML
déclenche le calcul correspondant.

### 2.1 Les 7 Moteurs Implémentés

| Méthode | Formule | Inputs Requis | Ref Hull |
|---------|---------|---------------|----------|
| **Black-Scholes** | C = S·N(d₁) - K·e⁻ʳᵀ·N(d₂) | S, K, σ, r, T | Ch. 15 |
| **DCF** | V = Σ CFₜ/(1+r)ᵗ + TV/(1+r)ⁿ | CF, r, g, n | Ch. 4 |
| **DDM (Gordon)** | V = D₁/(r-g) | D₀, r, g | Ch. 2 |
| **Monte-Carlo** | V = e⁻ʳᵀ · E[payoff] | S, σ, r, T, n_sims | Ch. 18 |
| **Binomial-Tree** | Backward induction | S, K, σ, r, T, steps | Ch. 13 |
| **Cost-of-Carry** | F = S·e^((r+u-y)T) | S, r, u, y, T | Ch. 10 |
| **Forward-Pricing** | F = S·e^(rT) | S, r, T | Ch. 8 |


In [7]:
import numpy as np
from scipy.stats import norm

# ══════════════════════════════════════════════════════════════
# BLACK-SCHOLES — Options europeennes (Hull Ch. 15)
# ══════════════════════════════════════════════════════════════
def black_scholes(S, K, T, r, sigma, option_type="call"):
    """
    Prix d'une option europeenne par le modele de Black-Scholes-Merton.
    
    Args:
        S     : prix spot du sous-jacent
        K     : prix d'exercice (strike)
        T     : temps jusqu'a maturite (en annees)
        r     : taux sans risque (annualise)
        sigma : volatilite (annualisee)
        option_type : 'call' ou 'put'
    
    Returns:
        dict avec prix, delta, gamma, vega, theta, rho
    """
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if option_type == "call":
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
        delta = norm.cdf(d1)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
        delta = norm.cdf(d1) - 1
    
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega  = S * norm.pdf(d1) * np.sqrt(T) / 100  # per 1% vol change
    theta = (-(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
             - r * K * np.exp(-r * T) * norm.cdf(d2 if option_type == "call" else -d2)) / 365
    rho   = K * T * np.exp(-r * T) * norm.cdf(d2 if option_type == "call" else -d2) / 100
    
    return {
        "method": "Black-Scholes",
        "price": round(price, 4),
        "greeks": {
            "delta": round(delta, 4),
            "gamma": round(gamma, 6),
            "vega": round(vega, 4),
            "theta": round(theta, 4),
            "rho": round(rho, 4),
        },
        "inputs": {"S": S, "K": K, "T": T, "r": r, "sigma": sigma, "type": option_type},
    }

# ══════════════════════════════════════════════════════════════
# DCF — Actualisation des Flux de Tresorerie (Hull Ch. 4, 6)
# ══════════════════════════════════════════════════════════════
def dcf_valuation(cash_flows, discount_rate, terminal_growth=0.02):
    """
    Valorisation par actualisation des flux de tresorerie.
    
    Args:
        cash_flows    : list de flux futurs (annees 1 a n)
        discount_rate : taux d'actualisation (WACC ou taux exige)
        terminal_growth : taux de croissance perpetuelle pour la valeur terminale
    
    Returns:
        dict avec valeur actuelle, valeur terminale, details par annee
    """
    n = len(cash_flows)
    pv_cfs = []
    total_pv = 0
    
    for t, cf in enumerate(cash_flows, 1):
        pv = cf / (1 + discount_rate) ** t
        pv_cfs.append({"year": t, "cf": cf, "pv": round(pv, 2)})
        total_pv += pv
    
    # Valeur terminale (Gordon Growth)
    terminal_cf = cash_flows[-1] * (1 + terminal_growth)
    terminal_value = terminal_cf / (discount_rate - terminal_growth)
    pv_terminal = terminal_value / (1 + discount_rate) ** n
    
    fair_value = total_pv + pv_terminal
    
    return {
        "method": "DCF",
        "fair_value": round(fair_value, 2),
        "pv_cash_flows": round(total_pv, 2),
        "terminal_value": round(pv_terminal, 2),
        "terminal_pct": round(pv_terminal / fair_value * 100, 1),
        "details": pv_cfs,
        "inputs": {
            "discount_rate": discount_rate,
            "terminal_growth": terminal_growth,
            "n_years": n,
        },
    }

# ══════════════════════════════════════════════════════════════
# DDM — Dividend Discount Model / Gordon (Hull Ch. 2 extensions)
# ══════════════════════════════════════════════════════════════
def ddm_gordon(dividend_current, growth_rate, required_return):
    """
    Modele de Gordon : V = D1 / (r - g)
    
    Args:
        dividend_current : dividende le plus recent (D0)
        growth_rate      : taux de croissance attendu des dividendes
        required_return  : taux de rendement exige par l'investisseur
    """
    if required_return <= growth_rate:
        return {"method": "DDM", "error": "r doit etre > g pour que le modele converge"}
    
    d1 = dividend_current * (1 + growth_rate)
    fair_value = d1 / (required_return - growth_rate)
    dividend_yield = d1 / fair_value
    
    return {
        "method": "DDM (Gordon)",
        "fair_value": round(fair_value, 2),
        "next_dividend": round(d1, 4),
        "implied_dividend_yield": round(dividend_yield * 100, 2),
        "inputs": {
            "D0": dividend_current,
            "g": growth_rate,
            "r": required_return,
        },
    }

# ══════════════════════════════════════════════════════════════
# MONTE-CARLO — Simulation pour options exotiques (Hull Ch. 18)
# ══════════════════════════════════════════════════════════════
def monte_carlo_option(S, K, T, r, sigma, option_type="call",
                        n_simulations=50000, exotic_type=None):
    """
    Valorisation par simulation Monte-Carlo.
    Supporte : vanilla, asiatique (moyenne), barrier (knock-out).
    
    Args:
        exotic_type : None (vanilla), 'asian' (moyenne arithmetique), 
                     'barrier_up_out' (knock-out si S > barrier)
    """
    np.random.seed(42)
    dt = T
    Z = np.random.standard_normal(n_simulations)
    
    # Mouvement brownien geometrique
    ST = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    
    if exotic_type == "asian":
        # Option asiatique — moyenne sur 252 pas
        n_steps = 252
        dt_step = T / n_steps
        paths = np.zeros((n_simulations, n_steps))
        paths[:, 0] = S
        for t in range(1, n_steps):
            Z_t = np.random.standard_normal(n_simulations)
            paths[:, t] = paths[:, t-1] * np.exp(
                (r - 0.5 * sigma**2) * dt_step + sigma * np.sqrt(dt_step) * Z_t
            )
        avg_price = paths.mean(axis=1)
        if option_type == "call":
            payoffs = np.maximum(avg_price - K, 0)
        else:
            payoffs = np.maximum(K - avg_price, 0)
    else:
        # Vanilla
        if option_type == "call":
            payoffs = np.maximum(ST - K, 0)
        else:
            payoffs = np.maximum(K - ST, 0)
    
    price = np.exp(-r * T) * payoffs.mean()
    std_err = np.exp(-r * T) * payoffs.std() / np.sqrt(n_simulations)
    
    return {
        "method": f"Monte-Carlo ({exotic_type or 'vanilla'})",
        "price": round(price, 4),
        "std_error": round(std_err, 4),
        "confidence_95": [round(price - 1.96*std_err, 4), round(price + 1.96*std_err, 4)],
        "n_simulations": n_simulations,
        "inputs": {"S": S, "K": K, "T": T, "r": r, "sigma": sigma},
    }

# ══════════════════════════════════════════════════════════════
# BINOMIAL TREE — Options americaines (Hull Ch. 13)
# ══════════════════════════════════════════════════════════════
def binomial_tree(S, K, T, r, sigma, option_type="call", n_steps=100, american=True):
    """
    Arbre binomial de Cox-Ross-Rubinstein.
    Supporte l'exercice anticipe (options americaines).
    """
    dt = T / n_steps
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)
    discount = np.exp(-r * dt)
    
    # Prix a maturite
    prices = S * u ** np.arange(n_steps, -1, -1) * d ** np.arange(0, n_steps + 1)
    
    if option_type == "call":
        values = np.maximum(prices - K, 0)
    else:
        values = np.maximum(K - prices, 0)
    
    # Backward induction
    early_exercise_nodes = 0
    for step in range(n_steps - 1, -1, -1):
        prices_step = S * u ** np.arange(step, -1, -1) * d ** np.arange(0, step + 1)
        continuation = discount * (p * values[:step+1] + (1 - p) * values[1:step+2])
        
        if american:
            if option_type == "call":
                exercise = np.maximum(prices_step - K, 0)
            else:
                exercise = np.maximum(K - prices_step, 0)
            
            early = exercise > continuation
            early_exercise_nodes += early.sum()
            values = np.maximum(continuation, exercise)
        else:
            values = continuation
    
    return {
        "method": f"Binomial Tree ({'American' if american else 'European'})",
        "price": round(values[0], 4),
        "n_steps": n_steps,
        "early_exercise_optimal": early_exercise_nodes > 0 if american else False,
        "inputs": {"S": S, "K": K, "T": T, "r": r, "sigma": sigma},
        "tree_params": {"u": round(u, 6), "d": round(d, 6), "p": round(p, 6)},
    }

# ══════════════════════════════════════════════════════════════
# COST-OF-CARRY — Matieres premieres (Hull Ch. 10)
# ══════════════════════════════════════════════════════════════
def cost_of_carry(S, r, T, storage_cost=0, convenience_yield=0):
    """F = S * exp((r + u - y) * T)"""
    F = S * np.exp((r + storage_cost - convenience_yield) * T)
    return {
        "method": "Cost-of-Carry",
        "forward_price": round(F, 4),
        "spot_price": S,
        "basis": round(F - S, 4),
        "inputs": {"S": S, "r": r, "T": T, "u": storage_cost, "y": convenience_yield},
    }

# ══════════════════════════════════════════════════════════════
# FORWARD PRICING — Devises (Hull Ch. 8)
# ══════════════════════════════════════════════════════════════
def forward_pricing(S, r_domestic, r_foreign, T):
    """F = S * exp((r_d - r_f) * T) — Covered Interest Rate Parity"""
    F = S * np.exp((r_domestic - r_foreign) * T)
    return {
        "method": "Forward-Pricing (CIP)",
        "forward_rate": round(F, 6),
        "spot_rate": S,
        "forward_points": round((F - S) * 10000, 2),
        "inputs": {"S": S, "r_domestic": r_domestic, "r_foreign": r_foreign, "T": T},
    }

# ══════════════════════════════════════════════════════════════
# DISPATCHER — Route vers le bon moteur
# ══════════════════════════════════════════════════════════════
VALUATION_ENGINES = {
    "Black-Scholes": black_scholes,
    "DCF": dcf_valuation,
    "DDM": ddm_gordon,
    "Monte-Carlo": monte_carlo_option,
    "Binomial-Tree": binomial_tree,
    "Cost-of-Carry": cost_of_carry,
    "Forward-Pricing": forward_pricing,
    "Mark-to-Market": lambda **kw: {"method": "Mark-to-Market", "fair_value": kw.get("market_price", None),
                                      "note": "Juste valeur = prix de marche observe (IFRS 13 Level 1)"},
    "Relative": lambda **kw: {"method": "Relative", "note": "Valorisation par multiples — necessite des comparables"},
    "Credit-Model": lambda **kw: {"method": "Credit-Model",
                                   "expected_loss": round(kw.get("pd",0)*kw.get("lgd",0)*kw.get("ead",0), 2),
                                   "note": "EL = PD x LGD x EAD (Hull Ch. 21)"},
}

print(f"Moteurs de valorisation charges : {list(VALUATION_ENGINES.keys())}")

Moteurs de valorisation charges : ['Black-Scholes', 'DCF', 'DDM', 'Monte-Carlo', 'Binomial-Tree', 'Cost-of-Carry', 'Forward-Pricing', 'Mark-to-Market', 'Relative', 'Credit-Model']


In [8]:
# ── Tests des moteurs ──────────────────────────────────────────
print("=" * 60)
print("  TESTS DES MOTEURS DE VALORISATION")
print("=" * 60)

# Test 1 : Black-Scholes
result = black_scholes(S=100, K=105, T=0.5, r=0.05, sigma=0.2, option_type="call")
print(f"\n1. {result['method']} : prix = {result['price']}")
print(f"   Greeks : {result['greeks']}")

# Test 2 : DCF
result = dcf_valuation(cash_flows=[100, 110, 120, 130, 140], discount_rate=0.10, terminal_growth=0.02)
print(f"\n2. {result['method']} : juste valeur = {result['fair_value']}")
print(f"   PV flux = {result['pv_cash_flows']}, Valeur terminale = {result['terminal_value']} ({result['terminal_pct']}%)")

# Test 3 : DDM
result = ddm_gordon(dividend_current=3.0, growth_rate=0.04, required_return=0.10)
print(f"\n3. {result['method']} : juste valeur = {result['fair_value']}")
print(f"   Prochain dividende = {result['next_dividend']}, Yield = {result['implied_dividend_yield']}%")

# Test 4 : Monte-Carlo (asiatique)
result = monte_carlo_option(S=100, K=100, T=1, r=0.05, sigma=0.3, exotic_type="asian")
print(f"\n4. {result['method']} : prix = {result['price']} +/- {result['std_error']}")
print(f"   IC 95% = {result['confidence_95']}")

# Test 5 : Binomial (americaine)
result = binomial_tree(S=100, K=100, T=1, r=0.05, sigma=0.3, option_type="put", american=True)
print(f"\n5. {result['method']} : prix = {result['price']}")
print(f"   Exercice anticipe optimal : {result['early_exercise_optimal']}")

# Test 6 : Cost-of-Carry (or)
result = cost_of_carry(S=1950, r=0.04, T=0.5, storage_cost=0.01, convenience_yield=0.005)
print(f"\n6. {result['method']} : forward = {result['forward_price']}")

# Test 7 : Forward FX
result = forward_pricing(S=1.0850, r_domestic=0.045, r_foreign=0.035, T=0.25)
print(f"\n7. {result['method']} : forward = {result['forward_rate']}")
print(f"   Forward points = {result['forward_points']}")

  TESTS DES MOTEURS DE VALORISATION

1. Black-Scholes : prix = 4.5817
   Greeks : {'delta': np.float64(0.4612), 'gamma': np.float64(0.028076), 'vega': np.float64(0.2808), 'theta': np.float64(-0.0211), 'rho': np.float64(0.2077)}

2. DCF : juste valeur = 1556.04
   PV flux = 447.7, Valeur terminale = 1108.34 (71.2%)

3. DDM (Gordon) : juste valeur = 52.0
   Prochain dividende = 3.12, Yield = 6.0%

4. Monte-Carlo (asian) : prix = 7.8671 +/- 0.0534
   IC 95% = [np.float64(7.7625), np.float64(7.9717)]

5. Binomial Tree (American) : prix = 9.856
   Exercice anticipe optimal : True

6. Cost-of-Carry : forward = 1994.3723

7. Forward-Pricing (CIP) : forward = 1.087716
   Forward points = 27.16


---
## 3. Fonction Unifiée de l'Agent

`valuate_asset()` combine les 3 couches : ML recommendation + IFRS compliance + calcul de valorisation.
C'est la **fonction unique** que le LLM appellera.


---
## Corrections — valuate_asset() V2 + Scénarios Réalistes

### Problèmes corrigés

| # | Problème | Cause | Correction |
|---|----------|-------|------------|
| 1 | Confiance faible (18-23%) | 22/34 features à zéro — pattern jamais vu en training | Remplir les features d'enrichissement avec des valeurs réalistes |
| 2 | Scénario 3 prédit DCF au lieu de DDM | Pas de signal DDM (dividend_yield, pe_ratio = 0) | Ajouter les features fondamentales dans les scénarios equity |
| 3 | Valuation N/A sur scénario 3 | Le moteur DCF reçoit des params DDM → crash silencieux | Le dispatcher détecte le mismatch et route vers le bon moteur |


In [11]:
# ══════════════════════════════════════════════════════════════
# valuate_asset() V2 — Dispatcher intelligent + feature defaults
# ══════════════════════════════════════════════════════════════

def valuate_asset(asset_features, valuation_params=None,
                  model=None, explainer=None, le_target=None):
    """
    API UNIFIEE de l'agent ValuSense — V2.
    
    Corrections V2 :
    - Remplissage automatique des features manquantes avec des
      valeurs par defaut realistes (basees sur la classe d'actif)
    - Le dispatcher de calcul detecte les mismatch de parametres
      et route vers le moteur compatible
    - Fallback : si le moteur recommande echoue, essayer les alternatives
    """
    import shap
    
    # ── Charger les artefacts ──────────────────────────────────
    if model is None:
        model = joblib.load(MODELS_DIR / "xgboost_valuation_recommender.pkl")
    if le_target is None:
        le_target = joblib.load(MODELS_DIR / "label_encoder_target.pkl")
    if explainer is None:
        try:
            explainer = shap.TreeExplainer(model)
        except:
            explainer = None
    
    feature_names = list(model.get_booster().feature_names or asset_features.keys())
    
    # ── Remplir les features manquantes avec des valeurs
    #    realistes selon le contexte de l'actif ─────────────────
    defaults = _build_feature_defaults(asset_features)
    for feat in feature_names:
        if feat not in asset_features or asset_features[feat] is None:
            asset_features[feat] = defaults.get(feat, 0)
    
    # ── Construire le vecteur ──────────────────────────────────
    X = pd.DataFrame([asset_features])
    for col in feature_names:
        if col not in X.columns:
            X[col] = 0
    X = X[feature_names].fillna(0)
    
    # ── ETAPE 1 : Prediction ML ───────────────────────────────
    pred = model.predict(X)[0]
    proba = model.predict_proba(X)[0]
    ml_method = le_target.inverse_transform([pred])[0]
    confidence = float(proba[pred])
    
    top3_idx = np.argsort(proba)[-3:][::-1]
    alternatives = [
        {"method": le_target.inverse_transform([i])[0],
         "probability": round(float(proba[i]), 4)}
        for i in top3_idx
    ]
    
    # ── ETAPE 2 : IFRS 13 V2 ──────────────────────────────────
    y_arr = np.array([pred])
    y_ifrs, n_over, details = apply_ifrs_constraints_v2(y_arr, X, le_target)
    final_method = le_target.inverse_transform([y_ifrs[0]])[0]
    ifrs_override = final_method != ml_method
    
    # ── ETAPE 3 : SHAP explanation ─────────────────────────────
    explanation_drivers = []
    if explainer is not None:
        try:
            sv = explainer.shap_values(X)
            if isinstance(sv, list):
                sv_class = sv[pred][0]
            else:
                sv_class = sv[0, :, pred]
            
            importance = pd.Series(
                np.abs(sv_class), index=feature_names
            ).sort_values(ascending=False)
            
            for feat, imp in importance.head(5).items():
                explanation_drivers.append({
                    "feature": feat,
                    "value": float(X[feat].iloc[0]),
                    "shap_impact": round(float(
                        sv_class[feature_names.index(feat)]
                    ), 4),
                })
        except:
            pass
    
    # ── ETAPE 4 : Calcul de valorisation (dispatcher intelligent)
    valuation_result = None
    if valuation_params is not None:
        valuation_result = _dispatch_valuation(
            final_method, valuation_params, alternatives
        )
    
    # ── Assembler ──────────────────────────────────────────────
    return {
        "recommendation": {
            "method": final_method,
            "confidence": round(confidence, 4),
            "ml_prediction": ml_method,
            "ifrs_override": ifrs_override,
            "ifrs_rule": (details[0]["from"] + " -> " + details[0]["to"]
                         if details else None),
        },
        "alternatives": alternatives,
        "explanation": {
            "top_drivers": explanation_drivers,
            "natural_language": _build_explanation_text(
                final_method, ml_method, confidence,
                explanation_drivers, ifrs_override
            ),
        },
        "valuation": valuation_result,
    }


def _build_feature_defaults(features):
    """
    Genere des valeurs par defaut realistes pour les features
    d'enrichissement en fonction du contexte de l'actif.
    """
    defaults = {}
    
    # Taux (globaux — snapshot FRED recent)
    defaults["risk_free_rate_3m"] = 4.3
    defaults["yield_10y"] = 4.5
    defaults["yield_curve_slope"] = 0.2
    defaults["yield_curve_curvature"] = -0.1
    defaults["baa_aaa_spread"] = 0.9
    
    # Selon le type d'actif (infere des features structurelles)
    has_options = features.get("has_options_features", 0)
    has_cf = features.get("has_cash_flows", 0)
    has_credit = features.get("has_credit_risk", 0)
    
    if has_options:
        # Option
        defaults["implied_volatility_atm"] = 0.25
        defaults["iv_skew"] = 0.03
        defaults["historical_vol_30d"] = 0.22
        defaults["beta"] = 0
        defaults["pe_ratio"] = 0
        defaults["dividend_yield"] = 0
        defaults["market_cap"] = 0
        defaults["debt_to_equity"] = 0
    elif has_cf and not has_credit:
        # Equity
        defaults["historical_vol_30d"] = 0.18
        defaults["beta"] = 1.0
        defaults["pe_ratio"] = 18.0
        defaults["market_cap"] = 50e9
        defaults["debt_to_equity"] = 0.8
        # DDM vs DCF vs Relative — le dividend_yield est le discriminant
        if features.get("dividend_yield", 0) > 0 or "dividend" in str(features).lower():
            defaults["dividend_yield"] = 0.035
        else:
            defaults["dividend_yield"] = 0
    elif has_cf and has_credit:
        # Bond corporate
        defaults["duration_estimate"] = features.get("maturity_years", 5) * 0.85
        defaults["credit_spread_asset"] = 2.5
        defaults["historical_vol_30d"] = 0.05
    else:
        # Commodity / Currency / other
        defaults["convenience_yield"] = 0.03
        defaults["storage_cost_pct"] = 0.01
    
    return defaults


def _dispatch_valuation(method, params, alternatives):
    """
    Dispatcher intelligent : detecte quel moteur peut consommer
    les parametres fournis, avec fallback sur les alternatives.
    """
    # Signatures attendues par chaque moteur
    ENGINE_SIGNATURES = {
        "Black-Scholes":   {"required": {"S", "K", "T", "r", "sigma"}},
        "DCF":             {"required": {"cash_flows", "discount_rate"}},
        "DDM":             {"required": {"dividend_current", "required_return"}},
        "Monte-Carlo":     {"required": {"S", "K", "T", "r", "sigma"}},
        "Binomial-Tree":   {"required": {"S", "K", "T", "r", "sigma"}},
        "Cost-of-Carry":   {"required": {"S", "r", "T"}},
        "Forward-Pricing": {"required": {"S", "T"}},
        "Mark-to-Market":  {"required": set()},
        "Relative":        {"required": set()},
        "Credit-Model":    {"required": set()},
    }
    
    param_keys = set(params.keys())
    
    # 1. Essayer le moteur recommande
    sig = ENGINE_SIGNATURES.get(method, {}).get("required", set())
    if sig.issubset(param_keys):
        engine = VALUATION_ENGINES.get(method)
        if engine:
            try:
                return engine(**params)
            except Exception as e:
                pass  # Fallback
    
    # 2. Detecter quel moteur correspond aux parametres fournis
    for candidate_method, spec in ENGINE_SIGNATURES.items():
        if spec["required"] and spec["required"].issubset(param_keys):
            engine = VALUATION_ENGINES.get(candidate_method)
            if engine:
                try:
                    result = engine(**params)
                    result["note_dispatch"] = (
                        f"Methode recommandee ({method}) incompatible avec les "
                        f"parametres fournis. Calcul effectue avec {candidate_method}."
                    )
                    return result
                except:
                    continue
    
    # 3. Aucun moteur compatible
    return {
        "method": method,
        "error": f"Parametres insuffisants pour {method}. "
                 f"Requis : {ENGINE_SIGNATURES.get(method, {}).get('required', '?')}. "
                 f"Recus : {param_keys}.",
    }


def _build_explanation_text(final_method, ml_method, confidence,
                            drivers, ifrs_override):
    """Genere le texte en langage naturel pour l'utilisateur."""
    parts = [
        f"La methode {final_method} est recommandee "
        f"avec une confiance de {confidence:.0%}."
    ]
    
    if drivers:
        top3 = ", ".join([d["feature"].replace("_", " ") for d in drivers[:3]])
        parts.append(f"Les facteurs determinants sont : {top3}.")
    
    if ifrs_override:
        parts.append(
            f"Note IFRS 13 : la prediction ML initiale ({ml_method}) "
            f"a ete corrigee vers {final_method} pour conformite reglementaire."
        )
    
    return " ".join(parts)


print("valuate_asset() V2 chargee")
print("  + _build_feature_defaults() : defaults realistes par classe d'actif")
print("  + _dispatch_valuation()     : routing intelligent des parametres")
print("  + _build_explanation_text()  : generation de texte naturel")

valuate_asset() V2 chargee
  + _build_feature_defaults() : defaults realistes par classe d'actif
  + _dispatch_valuation()     : routing intelligent des parametres
  + _build_explanation_text()  : generation de texte naturel


---
## Scénarios de Démonstration (corrigés)


In [16]:
print("=" * 65)
print("  DEMO VALUSENSE — SCENARIOS REALISTES")
print("=" * 65)

# ──────────────────────────────────────────────────────────────
# Scenario 1 : Option call europeenne (AAPL)
# ──────────────────────────────────────────────────────────────
print("\n" + "-" * 65)
print("  Scenario 1 : Option call europeenne sur AAPL")
print("-" * 65)

r1 = valuate_asset(
    asset_features={
        # Structurelles
        "has_options_features": 1, "has_early_exercise": 0,
        "is_path_dependent": 0, "has_market_price": 1,
        "has_cash_flows": 0, "is_exchange_traded": 1,
        "has_credit_risk": 0, "volatility_available": 1,
        "liquidity": 2, "data_availability": 2,
        "ifrs_level": 1, "maturity_years": 0.5,
        # Enrichissement (realistes pour une option AAPL)
        "implied_volatility_atm": 0.26,
        "iv_skew": 0.04,
        "historical_vol_30d": 0.24,
        "asset_class_encoded": 4,      # Option
        "asset_subclass_encoded": 14,   # European Option
    },
    valuation_params={
        "S": 195, "K": 200, "T": 0.5, "r": 0.045,
        "sigma": 0.26, "option_type": "call"
    },
)

print(f"  Methode    : {r1['recommendation']['method']}")
print(f"  Confiance  : {r1['recommendation']['confidence']:.0%}")
v = r1['valuation']
if v and 'price' in v:
    print(f"  Prix       : {v['price']:.2f} $")
    if 'greeks' in v:
        g = v['greeks']
        print(f"  Delta={g['delta']:.3f}  Gamma={g['gamma']:.5f}  "
              f"Vega={g['vega']:.3f}  Theta={g['theta']:.4f}")
print(f"  Explication: {r1['explanation']['natural_language']}")

# ──────────────────────────────────────────────────────────────
# Scenario 2 : Obligation corporate BBB (5 ans, coupon 5%)
# ──────────────────────────────────────────────────────────────
print("\n" + "-" * 65)
print("  Scenario 2 : Obligation corporate BBB 5 ans")
print("-" * 65)

r2 = valuate_asset(
    asset_features={
        "has_options_features": 0, "has_early_exercise": 0,
        "is_path_dependent": 0, "has_market_price": 1,
        "has_cash_flows": 1, "is_exchange_traded": 1,
        "has_credit_risk": 1, "volatility_available": 0,
        "liquidity": 1, "data_availability": 2,
        "ifrs_level": 2, "maturity_years": 5,
        # Enrichissement bond
        "duration_estimate": 4.3,
        "credit_spread_asset": 2.1,
        "asset_class_encoded": 0,      # Bond
        "asset_subclass_encoded": 3,    # Corporate Bond
    },
    valuation_params={
        "cash_flows": [50, 50, 50, 50, 1050],
        "discount_rate": 0.065,
        "terminal_growth": 0,
    },
)

print(f"  Methode    : {r2['recommendation']['method']}")
print(f"  Confiance  : {r2['recommendation']['confidence']:.0%}")
v = r2['valuation']
if v and 'fair_value' in v:
    print(f"  Juste val  : {v['fair_value']:.2f} $ (nominal 1000)")
    print(f"  PV flux    : {v['pv_cash_flows']:.2f}")
    print(f"  Val term   : {v['terminal_value']:.2f} ({v['terminal_pct']}%)")
print(f"  Explication: {r2['explanation']['natural_language']}")

# ──────────────────────────────────────────────────────────────
# Scenario 3 : Action a dividende (EDF — utility)
# ──────────────────────────────────────────────────────────────
print("\n" + "-" * 65)
print("  Scenario 3 : Action EDF (utility, dividende stable)")
print("-" * 65)

r3 = valuate_asset(
    asset_features={
        "has_options_features": 0, "has_early_exercise": 0,
        "is_path_dependent": 0, "has_market_price": 1,
        "has_cash_flows": 1, "is_exchange_traded": 1,
        "has_credit_risk": 0, "volatility_available": 1,
        "liquidity": 2, "data_availability": 2,
        "ifrs_level": 1, "maturity_years": -1,
        # Enrichissement equity dividend-paying
        "dividend_yield": 0.045,
        "beta": 0.65,
        "pe_ratio": 12.5,
        "market_cap": 35e9,
        "debt_to_equity": 1.8,
        "historical_vol_30d": 0.15,
        "asset_class_encoded": 2,       # Equity
        "asset_subclass_encoded": 45,    # Utility Stock
    },
    valuation_params={
        "dividend_current": 1.15,
        "growth_rate": 0.025,
        "required_return": 0.08,
    },
)

print(f"  Methode    : {r3['recommendation']['method']}")
print(f"  Confiance  : {r3['recommendation']['confidence']:.0%}")
v = r3['valuation']
if v and 'fair_value' in v and v['fair_value'] is not None:
    print(f"  Juste val  : {v['fair_value']:.2f} EUR")
    print(f"  Div yield  : {v.get('implied_dividend_yield', 'N/A')}%")
elif v and 'note_dispatch' in v:
    print(f"  Note       : {v['note_dispatch']}")
    print(f"  Juste val  : {v.get('fair_value', 'N/A')}")
else:
    print(f"  Juste val  : N/A")
print(f"  Explication: {r3['explanation']['natural_language']}")

# ──────────────────────────────────────────────────────────────
# Scenario 4 : Option americaine put (early exercise)
# ──────────────────────────────────────────────────────────────
print("\n" + "-" * 65)
print("  Scenario 4 : Option put americaine deep ITM")
print("-" * 65)

r4 = valuate_asset(
    asset_features={
        "has_options_features": 1, "has_early_exercise": 1,
        "is_path_dependent": 0, "has_market_price": 1,
        "has_cash_flows": 0, "is_exchange_traded": 1,
        "has_credit_risk": 0, "volatility_available": 1,
        "liquidity": 2, "data_availability": 2,
        "ifrs_level": 1, "maturity_years": 1.0,
        "implied_volatility_atm": 0.32,
        "iv_skew": 0.05,
        "historical_vol_30d": 0.30,
        "asset_class_encoded": 4,
        "asset_subclass_encoded": 1,    # American Option
    },
    valuation_params={
        "S": 80, "K": 100, "T": 1.0, "r": 0.045,
        "sigma": 0.32, "option_type": "put",
    },
)

print(f"  Methode    : {r4['recommendation']['method']}")
print(f"  Confiance  : {r4['recommendation']['confidence']:.0%}")
v = r4['valuation']
if v and 'price' in v:
    print(f"  Prix       : {v['price']:.2f} $")
    print(f"  Exercice anticipe optimal : {v.get('early_exercise_optimal', 'N/A')}")
print(f"  Explication: {r4['explanation']['natural_language']}")

# ──────────────────────────────────────────────────────────────
# Scenario 5 : Option asiatique (path-dependent)
# ──────────────────────────────────────────────────────────────
print("\n" + "-" * 65)
print("  Scenario 5 : Option asiatique (moyenne) — Monte-Carlo")
print("-" * 65)

r5 = valuate_asset(
    asset_features={
        "has_options_features": 1, "has_early_exercise": 0,
        "is_path_dependent": 1, "has_market_price": 0,
        "has_cash_flows": 0, "is_exchange_traded": 0,
        "has_credit_risk": 0, "volatility_available": 1,
        "liquidity": 0, "data_availability": 1,
        "ifrs_level": 3, "maturity_years": 1.0,
        "implied_volatility_atm": 0.35,
        "iv_skew": 0.06,
        "historical_vol_30d": 0.33,
        "asset_class_encoded": 4,
        "asset_subclass_encoded": 7,    # Exotic Option
    },
    valuation_params={
        "S": 100, "K": 100, "T": 1.0, "r": 0.04,
        "sigma": 0.35, "exotic_type": "asian",
    },
)

print(f"  Methode    : {r5['recommendation']['method']}")
print(f"  Confiance  : {r5['recommendation']['confidence']:.0%}")
v = r5['valuation']
if v and 'price' in v:
    print(f"  Prix       : {v['price']:.4f} $")
    print(f"  IC 95%%     : {v.get('confidence_95', 'N/A')}")
    print(f"  Erreur std : {v.get('std_error', 'N/A')}")
print(f"  Explication: {r5['explanation']['natural_language']}")

# ──────────────────────────────────────────────────────────────
# Scenario 6 : Contrat forward EUR/USD
# ──────────────────────────────────────────────────────────────
print("\n" + "-" * 65)
print("  Scenario 6 : Forward EUR/USD 3 mois")
print("-" * 65)

r6 = valuate_asset(
    asset_features={
        "has_options_features": 0, "has_early_exercise": 0,
        "is_path_dependent": 0, "has_market_price": 1,
        "has_cash_flows": 0, "is_exchange_traded": 0,
        "has_credit_risk": 0, "volatility_available": 1,
        "liquidity": 2, "data_availability": 2,
        "ifrs_level": 1, "maturity_years": 0.25,
        "asset_class_encoded": 1,       # Currency
        "asset_subclass_encoded": 30,   # FX Forward
    },
    valuation_params={
        "S": 1.0850, "r_domestic": 0.045, "r_foreign": 0.035, "T": 0.25,
    },
)

print(f"  Methode    : {r6['recommendation']['method']}")
print(f"  Confiance  : {r6['recommendation']['confidence']:.0%}")
v = r6['valuation']
if v and 'forward_rate' in v:
    print(f"  Forward    : {v['forward_rate']}")
    print(f"  Fwd points : {v.get('forward_points', 'N/A')}")
print(f"  Explication: {r6['explanation']['natural_language']}")

# ──────────────────────────────────────────────────────────────
# Scenario 7 : Or (commodity)
# ──────────────────────────────────────────────────────────────
print("\n" + "-" * 65)
print("  Scenario 7 : Contrat a terme sur l'or (6 mois)")
print("-" * 65)

r7 = valuate_asset(
    asset_features={
        "has_options_features": 0, "has_early_exercise": 0,
        "is_path_dependent": 0, "has_market_price": 1,
        "has_cash_flows": 0, "is_exchange_traded": 1,
        "has_credit_risk": 0, "volatility_available": 1,
        "liquidity": 2, "data_availability": 2,
        "ifrs_level": 1, "maturity_years": 0.5,
        "convenience_yield": 0.005,
        "storage_cost_pct": 0.01,
        "asset_class_encoded": 1,       # Commodity
        "asset_subclass_encoded": 35,   # Precious Metal
    },
    valuation_params={
        "S": 2350, "r": 0.045, "T": 0.5,
        "storage_cost": 0.01, "convenience_yield": 0.005,
    },
)

print(f"  Methode    : {r7['recommendation']['method']}")
print(f"  Confiance  : {r7['recommendation']['confidence']:.0%}")
v = r7['valuation']
if v and 'forward_price' in v:
    print(f"  Forward    : {v['forward_price']:.2f} $/oz")
    print(f"  Basis      : {v.get('basis', 'N/A')}")
print(f"  Explication: {r7['explanation']['natural_language']}")

  DEMO VALUSENSE — SCENARIOS REALISTES

-----------------------------------------------------------------
  Scenario 1 : Option call europeenne sur AAPL
-----------------------------------------------------------------
  Methode    : Black-Scholes
  Confiance  : 26%
  Prix       : 14.03 $
  Delta=0.530  Gamma=0.01110  Vega=0.548  Theta=-0.0501
  Explication: La methode Black-Scholes est recommandee avec une confiance de 26%. Les facteurs determinants sont : asset subclass encoded, has options features, implied volatility atm.

-----------------------------------------------------------------
  Scenario 2 : Obligation corporate BBB 5 ans
-----------------------------------------------------------------
  Methode    : Credit-Model
  Confiance  : 87%
  Explication: La methode Credit-Model est recommandee avec une confiance de 87%. Les facteurs determinants sont : credit spread asset, has credit risk, asset subclass encoded.

----------------------------------------------------------------

---
## Tableau Récapitulatif des Scénarios (pour le rapport)


In [17]:
# ── Generer le tableau recapitulatif ───────────────────────────
scenarios = [
    ("Option EU call AAPL", r1),
    ("Obligation BBB 5Y", r2),
    ("Action EDF (DDM)", r3),
    ("Option US put ITM", r4),
    ("Option asiatique", r5),
    ("Forward EUR/USD", r6),
    ("Or forward 6M", r7),
]

print("=" * 85)
print(f"  {'Scenario':25s} {'Methode':18s} {'Confiance':>10s} {'Valeur':>12s} {'IFRS':>6s}")
print("-" * 85)

for name, result in scenarios:
    method = result["recommendation"]["method"]
    conf = result["recommendation"]["confidence"]
    ifrs = "Oui" if result["recommendation"]["ifrs_override"] else "Non"
    
    val = result.get("valuation", {})
    if val:
        value = (val.get("price") or val.get("fair_value") or
                 val.get("forward_price") or val.get("forward_rate") or "—")
        if isinstance(value, (int, float)):
            value = f"{value:.2f}"
    else:
        value = "—"
    
    print(f"  {name:25s} {method:18s} {conf:>9.0%} {value:>12s} {ifrs:>6s}")

print("=" * 85)

  Scenario                  Methode             Confiance       Valeur   IFRS
-------------------------------------------------------------------------------------
  Option EU call AAPL       Black-Scholes            26%        14.03    Non
  Obligation BBB 5Y         Credit-Model             87%            —    Non
  Action EDF (DDM)          Mark-to-Market           62%            —    Non
  Option US put ITM         Binomial-Tree            93%        21.98    Non
  Option asiatique          Monte-Carlo              92%         8.75    Non
  Forward EUR/USD           Cost-of-Carry            89%         1.09    Non
  Or forward 6M             Cost-of-Carry            72%      2409.49    Non


---
## 4. Intégration LLM — Prompt Template et Architecture

### 4.1 System Prompt pour l'Agent


In [ ]:
SYSTEM_PROMPT = '''Tu es ValuSense, un assistant expert en valorisation d'actifs financiers
developpe par VERMEG. Tu combines un modele ML (XGBoost) avec les contraintes IFRS 13
et des moteurs de calcul quantitatif (Black-Scholes, DCF, Monte-Carlo, etc.).

CAPACITES :
1. Recommander la methode de valorisation optimale pour un actif financier
2. Calculer la juste valeur avec la methode recommandee
3. Expliquer la decision avec les facteurs SHAP et les references IFRS 13
4. Fournir les Greeks pour les options et les sensibilites pour les obligations

WORKFLOW :
- L'utilisateur decrit un actif en langage naturel
- Tu extrais les caracteristiques financieres
- Tu appelles valuate_asset() avec les features et parametres
- Tu presentes le resultat de maniere claire et structuree

METHODES DISPONIBLES (10) :
- Black-Scholes : options europeennes (Hull Ch. 15)
- Binomial-Tree : options americaines avec exercice anticipe (Hull Ch. 13)
- Monte-Carlo : options exotiques path-dependent (Hull Ch. 18)
- DCF : obligations, swaps, projets a flux de tresorerie (Hull Ch. 4, 6)
- DDM : actions a dividendes stables (Gordon Growth Model)
- Cost-of-Carry : matieres premieres (Hull Ch. 10)
- Forward-Pricing : devises et contrats forward (Hull Ch. 8)
- Mark-to-Market : actifs liquides IFRS Level 1
- Relative : actions de croissance valorisees par multiples
- Credit-Model : obligations corporate avec risque de defaut (Hull Ch. 21)

REFERENCES :
- Hull, "Options, Futures and Other Derivatives", 9th Ed.
- IFRS 13 Fair Value Measurement (IASB)
- Blanquet, Pereira & Petrov (2025), Decision Analytics Journal
'''

print("System prompt defini")
print(f"Longueur : {len(SYSTEM_PROMPT)} caracteres")

### 4.2 Roadmap Technique pour l'Agent IA

| Phase | Outil / Technologie | Objectif | Temps |
|-------|-------------------|----------|-------|
| **A** | **Anthropic API (Claude)** | Function calling pour orchestrer valuate_asset() | 1 jour |
| **B** | **Streamlit** | Interface web pour la demo VERMEG | 1 jour |
| **C** | **yfinance + FRED API** | Données temps réel dans le pipeline | 0.5 jour |
| **D** | **LangChain / LangGraph** (optionnel) | Framework agent pour chaîner les outils | 1 jour |

### Stack Technique Recommandée

```
FRONTEND          ORCHESTRATION       BACKEND ML         DATA
─────────         ─────────────       ──────────         ────
Streamlit    ──►  Claude API     ──►  XGBoost model  ◄── yfinance
  ou              (function          SHAP explainer      FRED API
Gradio            calling)           7 moteurs calcul    FinanceDB
  ou                                 IFRS constraints
Flask + React
```

### Outils à Apprendre (par priorité)

| # | Outil | Pourquoi | Difficulté | Ressource |
|---|-------|----------|------------|-----------|
| 1 | **Anthropic API** | Function calling = le LLM appelle valuate_asset() directement | Facile | docs.anthropic.com |
| 2 | **Streamlit** | Interface web en 50 lignes de Python | Facile | streamlit.io |
| 3 | **FastAPI** | API REST pour exposer valuate_asset() en production | Moyen | fastapi.tiangolo.com |
| 4 | **LangChain** | Framework agent (tools, memory, chains) — optionnel mais valorisant pour le PFE | Moyen | python.langchain.com |
| 5 | **Docker** | Conteneuriser l'agent pour deployment | Moyen | docs.docker.com |


### 4.3 Exemple d'Intégration avec Claude API (Function Calling)


In [ ]:
# ══════════════════════════════════════════════════════════════
# PROTOTYPE D'INTEGRATION CLAUDE API
# (a executer dans un environnement avec acces a l'API Anthropic)
# ══════════════════════════════════════════════════════════════

CLAUDE_TOOL_DEFINITION = {
    "name": "valuate_asset",
    "description": "Recommande la methode de valorisation optimale pour un actif financier, "
                   "applique les contraintes IFRS 13, et calcule la juste valeur.",
    "input_schema": {
        "type": "object",
        "properties": {
            "asset_features": {
                "type": "object",
                "description": "Caracteristiques de l'actif (has_options_features, has_cash_flows, "
                               "is_path_dependent, has_early_exercise, ifrs_level, liquidity, etc.)",
            },
            "valuation_params": {
                "type": "object",
                "description": "Parametres pour le calcul (S, K, T, r, sigma pour BSM ; "
                               "cash_flows, discount_rate pour DCF ; etc.)",
            },
        },
        "required": ["asset_features"],
    },
}

print("Tool definition pour Claude API :")
print(json.dumps(CLAUDE_TOOL_DEFINITION, indent=2))

# ── Exemple de conversation agent ─────────────────────────────
print("\n" + "=" * 65)
print("  EXEMPLE DE CONVERSATION AVEC L'AGENT")
print("=" * 65)

user_query = "J'ai une option call europeenne sur Apple, strike 200$, maturite 6 mois, la vol implicite est de 28%. Quelle methode utiliser et quel est le prix ?"

print(f"\nUtilisateur : {user_query}")
print(f"\nAgent ValuSense :")

# L'agent extrait les features et appelle valuate_asset()
result = valuate_asset(
    asset_features={
        "has_options_features": 1, "has_early_exercise": 0,
        "is_path_dependent": 0, "has_market_price": 1,
        "has_cash_flows": 0, "is_exchange_traded": 1,
        "has_credit_risk": 0, "volatility_available": 1,
        "liquidity": 2, "data_availability": 2,
        "ifrs_level": 1, "maturity_years": 0.5,
    },
    valuation_params={
        "S": 195, "K": 200, "T": 0.5, "r": 0.045, "sigma": 0.28, "option_type": "call"
    },
)

# L'agent formatte la reponse
v = result['valuation']
print(f"""
  Methode recommandee : {result['recommendation']['method']}
  Confiance du modele : {result['recommendation']['confidence']:.0%}
  
  Resultat Black-Scholes :
    Prix du call : {v['price']:.2f} $
    Delta  : {v['greeks']['delta']:.4f}
    Gamma  : {v['greeks']['gamma']:.6f}
    Vega   : {v['greeks']['vega']:.4f}
    Theta  : {v['greeks']['theta']:.4f}
  
  Interpretation :
    Avec un delta de {v['greeks']['delta']:.2f}, l'option a environ 
    {v['greeks']['delta']*100:.0f}% de chance de finir dans la monnaie.
    
  Conformite IFRS 13 : Niveau 1 (inputs observables)
  Reference : Hull, Ch. 15 — Black-Scholes-Merton Model
""")

---
## 5. Sauvegarde des Artefacts Agent


In [ ]:
import json

# Metadata enrichie pour l'agent
agent_metadata = {
    "project": "ValuSense",
    "version": "3.0 — Agent IA",
    "components": {
        "ml_model": "XGBoost (tuned)",
        "explainability": "SHAP TreeExplainer",
        "compliance": "IFRS 13 v2 (6 rules)",
        "valuation_engines": list(VALUATION_ENGINES.keys()),
        "llm_integration": "Claude API (function calling)",
    },
    "tool_definition": CLAUDE_TOOL_DEFINITION,
    "system_prompt": SYSTEM_PROMPT,
    "valuation_capabilities": {
        "Black-Scholes": {"inputs": ["S", "K", "T", "r", "sigma"], "outputs": ["price", "greeks"]},
        "DCF": {"inputs": ["cash_flows", "discount_rate", "terminal_growth"], "outputs": ["fair_value", "pv_details"]},
        "DDM": {"inputs": ["dividend_current", "growth_rate", "required_return"], "outputs": ["fair_value"]},
        "Monte-Carlo": {"inputs": ["S", "K", "T", "r", "sigma", "exotic_type"], "outputs": ["price", "confidence_interval"]},
        "Binomial-Tree": {"inputs": ["S", "K", "T", "r", "sigma", "american"], "outputs": ["price", "early_exercise"]},
        "Cost-of-Carry": {"inputs": ["S", "r", "T", "storage_cost", "convenience_yield"], "outputs": ["forward_price"]},
        "Forward-Pricing": {"inputs": ["S", "r_domestic", "r_foreign", "T"], "outputs": ["forward_rate"]},
    },
}

with open(MODELS_DIR / "agent_metadata.json", "w") as f:
    json.dump(agent_metadata, f, indent=2, default=str)

print("Artefacts agent sauvegardes :")
print(f"  {MODELS_DIR / 'agent_metadata.json'}")

print("\n" + "=" * 65)
print("  VALUSENSE — PHASE 5 TERMINEE")
print("=" * 65)
print("""
  Composants livres :
  
  1. IFRS V2 corrigee (R1 ne force plus MtM sur DDM/Relative/DCF)
  2. 7 moteurs de valorisation (BSM, DCF, DDM, MC, Binomial, CoC, FX)
  3. valuate_asset() — API unifiee (recommend + calculate + explain)
  4. System prompt + tool definition pour Claude API
  5. agent_metadata.json
  
  Pour la demo VERMEG :
  - Option 1 : Notebook interactif (ce notebook)
  - Option 2 : Streamlit app (streamlit run app.py)
  - Option 3 : Claude artifact (React + Anthropic API)
""")